# cvprofiles — measure discipline tour (v2.0 diagnostics)

This notebook exercises the full **v2.0 diagnostic stack** shipped in the published package:
all four restriction evaluators in one network (`corr_min`, `corr_sign`, `mean_order`,
`rank_agree`), a regression target functional (`ols_coef` with a control), and all four
additive diagnostics in a single run — **bootstrap** over units, the **θ-grid** sensitivity
surface, the **δ-grid** tolerance surface, and the **θ-anchor** pre-data audit.

It is a companion to the main tutorial. Everything here is generated inline — no repository
files, no data downloads. It runs against whatever `cvprofiles` is installed (`pip install cvprofiles`).

Scientific stance: **empty admissible sets and wide construct-identified ranges are findings,
not failures.** The headline range `[L,U]` is always the image of the target functional on the
survivors of the researcher-authored nomological network.


## 1. Environment

The only hard dependencies are the package's own: pandas, numpy, PyYAML.


In [ ]:
from __future__ import annotations
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import cvprofiles
from cvprofiles.pipeline import run_profile

print("cvprofiles", cvprofiles.__version__)

WORK = Path(tempfile.mkdtemp(prefix="cvp_tour_"))
print("work dir:", WORK)


## 2. Synthetic construct with a designed menu

A latent construct `C` drives an auxiliary correlate `v_aux1`, a known-group indicator `G`,
a reference measure `m_ref`, and an outcome `y`. The researcher's menu has five candidate
operationalizations:

| measure | design | intended fate |
|---|---|---|
| `m_good` | strong loading on `C` | admissible |
| `m_weak` | moderate loading on `C` | admissible (barely) — drives the wide range |
| `m_groupy` | loads on `C` **and** discriminates `G` | admissible |
| `m_wrong_sign` | loads on `-C` | rejected (wrong sign) |
| `m_noise` | pure noise | rejected (fails every restriction) |

The point of `m_weak`: both `m_good` and `m_weak` pass the network, but their regression
coefficients differ a lot — so `[L,U]` is **wide**, quantifying measurement fragility.


In [ ]:
rng = np.random.default_rng(20260806)
n = 300

C = rng.normal(size=n)                       # latent construct
v_aux1 = 0.9 * C + 0.4 * rng.normal(size=n)  # auxiliary correlate
z_control = rng.normal(size=n)               # control for beta
G = (C + rng.normal(size=n) > 0).astype(float)      # known-group indicator, correlated with C
m_ref = 0.85 * C + 0.6 * rng.normal(size=n)  # reference measure (for rank_agree)
y = 0.6 * C + 0.3 * z_control + rng.normal(size=n)

def e():
    return rng.normal(size=n)

m_good       = 0.90 * C + 0.40 * e()
m_weak       = 0.50 * C + 0.87 * e()
m_groupy     = 0.70 * C + 0.80 * G + 0.50 * e()
m_wrong_sign = -0.50 * C + 0.87 * e()
m_noise      = 1.00 * e()

scores = pd.DataFrame({
    "unit_id": [f"u{i:03d}" for i in range(n)],
    "m_good": m_good, "m_weak": m_weak, "m_groupy": m_groupy,
    "m_wrong_sign": m_wrong_sign, "m_noise": m_noise,
    "v_aux1": v_aux1, "m_ref": m_ref, "G": G, "z_control": z_control,
    "y": y,
})

roles = {
    "unit_id": "unit_id",
    "measures": ["m_good", "m_weak", "m_groupy", "m_wrong_sign", "m_noise"],
    "aux": ["v_aux1", "m_ref", "G", "z_control"],
    "outcome": "y",
    "diagnostic": [],
}

# Nomological network R: one restriction of each implemented type.
#   corr_min    : Corr(m, v_aux1) >= 0.35
#   corr_sign   : Corr(m, v_aux1) has sign +1 with magnitude >= 0.10
#   mean_order  : mean(m | G=1) - mean(m | G=0) >= 0.10
#   rank_agree  : Spearman(m, m_ref) >= 0.35
network = {
    "schema_version": "1",
    "name": "tour_synthetic",
    "delta": 0.0,
    "restrictions": [
        {"id": "r_corr_min_aux1",    "type": "corr_min",   "theta": 0.35, "params": {"variable": "v_aux1"}},
        {"id": "r_corr_sign_aux1",   "type": "corr_sign",  "theta": 0.10, "params": {"variable": "v_aux1", "sign": 1}},
        {"id": "r_mean_order_group", "type": "mean_order", "theta": 0.10, "params": {"group": "G", "sign": 1}},
        {"id": "r_rank_agree_ref",   "type": "rank_agree", "theta": 0.35, "params": {"ref_measure": "m_ref"}},
    ],
}

# Target functional: standardized OLS coefficient on the measure, controlling z_control.
beta = {"schema_version": "1", "type": "ols_coef", "outcome": "y", "params": {"controls": ["z_control"]}}

# Pre-data theta anchors: documentation provenance, one anchor per restriction.
anchors = {
    "schema_version": "1",
    "anchors": [
        {"restriction_id": "r_corr_min_aux1",    "citation_key": "tour2026", "source_phrase": "Synthetic tour: min corr with aux 0.35 (illustrative pre-data anchor)", "anchor_kind": "author", "pre_data": True},
        {"restriction_id": "r_corr_sign_aux1",   "citation_key": "tour2026", "source_phrase": "Synthetic tour: positive sign vs aux (illustrative)", "anchor_kind": "author", "pre_data": True},
        {"restriction_id": "r_mean_order_group", "citation_key": "tour2026", "source_phrase": "Synthetic tour: known-group gap 0.10 (illustrative)", "anchor_kind": "author", "pre_data": True},
        {"restriction_id": "r_rank_agree_ref",   "citation_key": "tour2026", "source_phrase": "Synthetic tour: rank agreement vs ref 0.35 (illustrative)", "anchor_kind": "author", "pre_data": True},
    ],
}

scores.to_csv(WORK / "scores.csv", index=False)
(WORK / "roles.json").write_text(json.dumps(roles))
(WORK / "network.yaml").write_text(yaml.safe_dump(network))
(WORK / "beta.yaml").write_text(yaml.safe_dump(beta))
(WORK / "anchors.yaml").write_text(yaml.safe_dump(anchors))
print("inputs written under", WORK)


## 3. Headline run with every diagnostic layer on

One `run_profile` call: bootstrap over units (`n_boot=200`), the θ-grid
(`λ = 0.5, 1.0, 1.5, 2.0, 2.5`), the δ-grid (`δ = 0.0, 0.2, 0.5`), and the anchors
audit. All diagnostics are **additive** — the headline `[L,U]` is untouched.


In [ ]:
result = run_profile(
    scores=WORK / "scores.csv",
    roles=WORK / "roles.json",
    network=WORK / "network.yaml",
    beta=WORK / "beta.yaml",
    anchors=WORK / "anchors.yaml",
    out_dir=WORK / "run_full",
    seed=0,
    title="Diagnostics tour",
    n_boot=200,
    theta_grid_lambdas=[0.5, 1.0, 1.5, 2.0, 2.5],
    delta_grid_deltas=[0.0, 0.5, 0.9],
)

print("run_id      :", result.run_id)
print("M*          :", result.identify.admissible)
print("[L,U]       :", result.identify.range_L, result.identify.range_U)
print("point_id    :", result.identify.point_id)
print("anchors_hash:", result.anchors_hash)
print("report html :", result.report.html_path)


### Slack matrix and the beta image

`slacks.csv` is the heart of the audit trail: measure × restriction sample slacks.
A measure is admissible iff every slack is `>= -delta`. `beta_values.json` reports
β for **all** menu measures; non-survivors are marked and never enter `[L,U]`.


In [ ]:
print("slack matrix (admit iff slack >= -delta)")
print(result.identify.slacks.round(3).to_string())

print()
beta_df = pd.DataFrame({
    "beta": pd.Series(result.identify.beta_values),
    "admissible": [m in result.identify.admissible for m in result.identify.measures],
}).round(4)
print("beta image (ols_coef on y, controlling z_control)")
print(beta_df.to_string())


### Reading the wide range

`m_good` (β ≈ 0.55) and `m_weak` (β ≈ 0.30) are **both** admissible under the network,
so the construct-identified range `[L,U]` is wide. That is not a bug: under this theory
and data, the measurement conclusion is genuinely fragile. A singleton `M*` would give
point identification *under this network* — nothing more.

## 4. θ-grid: what happens when the thresholds tighten?

λ scales every threshold magnitude (sign/direction constraints and δ are **never** scaled).
λ=1.0 is the declared network. Watch `m_weak` — the marginal measure — drop out, then the
set empty.


In [ ]:
rows = result.theta_grid.rows
tg = pd.DataFrame([{
    "lambda": r.lambda_value,
    "M*": ", ".join(r.admissible) if r.admissible else "empty",
    "n": r.n_admissible,
    "L": None if r.range_L is None else round(r.range_L, 4),
    "U": None if r.range_U is None else round(r.range_U, 4),
    "point_id": r.point_id,
} for r in rows])
print("theta-grid (lambda scales all thresholds; lambda=1.0 is the declared network)")
print(tg.to_string(index=False))


## 5. δ-grid: how much slack do we tolerate?

δ is an **absolute** tolerance on slacks (`admit iff slack >= -δ`). As δ grows, `M*` is a
monotone superset. At δ=0.5 the noise measure creeps in; at δ=0.9 even the wrong-sign measure
passes and the range collapses toward β ≈ 0. That is the honest face of tolerance: loosen it
enough and measurement discipline evaporates. The headline stays at the declared δ=0.0.


In [ ]:
rows = result.delta_grid.rows
dg = pd.DataFrame([{
    "delta": r.delta_value,
    "M*": ", ".join(r.admissible) if r.admissible else "empty",
    "n": r.n_admissible,
    "L": None if r.range_L is None else round(r.range_L, 4),
    "U": None if r.range_U is None else round(r.range_U, 4),
} for r in rows])
print("delta-grid (absolute slack tolerance; delta=0.0 is the declared network)")
print(dg.to_string(index=False))


## 6. Bootstrap: sampling variation of the range

Units-only resampling (the menu is fixed). Percentile band over **non-empty** replicates;
empty and degenerate replicates are counted and reported, never silently dropped. The band is
additive metadata — it never replaces the headline `[L,U]`.


In [ ]:
b = result.bootstrap
print("bootstrap (units-only, seed", b.seed_used, ", n_boot", b.n_boot, ")")
print("headline [L,U]     :", round(result.identify.range_L, 4), round(result.identify.range_U, 4))
print("band [L,U]         :", None if b.band_L is None else round(b.band_L, 4),
      None if b.band_U is None else round(b.band_U, 4))
print("nonempty/empty/degenerate:", b.replicates_nonempty, b.replicates_empty, b.replicates_degenerate)
print("empty_replicate_rate      :", round(b.empty_replicate_rate, 3))
print("note:", b.note)


## 7. θ-anchors: documentation provenance with teeth

Anchors are a schema'd, completeness-checked pre-data file: one anchor per restriction,
`pre_data: true`. They are **excluded from the freeze preimage** — same bundle ± anchors
⇒ same `run_id`, different `anchors_hash` / `anchors.json`. But the file is enforced:
an incomplete anchor set **fails loud** instead of silently proceeding.


In [ ]:
# Same bundle WITHOUT anchors: run_id must be unchanged.
result_no_anchors = run_profile(
    scores=WORK / "scores.csv",
    roles=WORK / "roles.json",
    network=WORK / "network.yaml",
    beta=WORK / "beta.yaml",
    out_dir=WORK / "run_no_anchors",
    seed=0,
    n_boot=200,
)
print("run_id with anchors   :", result.run_id)
print("run_id without anchors:", result_no_anchors.run_id)
print("same run_id           :", result_no_anchors.run_id == result.run_id)
print("anchors_hash (with)   :", result.anchors_hash)

# Incomplete anchors must fail loud at completeness check.
incomplete = {**anchors, "anchors": anchors["anchors"][:2]}
(WORK / "anchors_incomplete.yaml").write_text(yaml.safe_dump(incomplete))

from cvprofiles.anchors.pipeline import AnchorError
try:
    run_profile(
        scores=WORK / "scores.csv",
        roles=WORK / "roles.json",
        network=WORK / "network.yaml",
        beta=WORK / "beta.yaml",
        anchors=WORK / "anchors_incomplete.yaml",
        out_dir=WORK / "run_bad",
        seed=0,
    )
    print("ERROR: expected AnchorError")
except AnchorError as exc:
    print("AnchorError (expected):", str(exc)[:120])


## 8. The CLI: one machine-clean JSON summary

`cvprofiles run` exposes the same composition with flags; **stdout is pure JSON**
(human status goes to stderr). Empty `M*` exits 0 — an empty set is a result, not a crash.


In [ ]:
cmd = [
    sys.executable, "-m", "cvprofiles", "run",
    "--scores", str(WORK / "scores.csv"),
    "--roles", str(WORK / "roles.json"),
    "--network", str(WORK / "network.yaml"),
    "--beta", str(WORK / "beta.yaml"),
    "--anchors", str(WORK / "anchors.yaml"),
    "--out", str(WORK / "run_cli"),
    "--seed", "0",
    "--n-boot", "100",
    "--theta-grid", "0.5,1.0,1.5",
    "--delta-grid", "0.0,0.2",
    "--title", "CLI diagnostics tour",
]
proc = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
print("exit code:", proc.returncode)
cli = json.loads(proc.stdout)  # stdout is pure JSON
print("M*      :", cli["M_star"])
print("[L,U]   :", cli["L"], cli["U"])
print("band    :", cli["bootstrap"]["band_L"], cli["bootstrap"]["band_U"])
print("theta-grid rows:", cli["theta_grid"]["rows"], "| delta-grid rows:", cli["delta_grid"]["rows"])
print("anchors_hash:", cli["anchors_hash"][:12], "...")


## 9. Assertions (the tour is self-checking)

These encode the package's core contracts: survivors only in the range, anchors excluded
from the run_id, monotone δ-grid, θ-grid empties under tightening, machine-clean CLI.


In [ ]:
# Designed-valid measures survive; designed invalids do not.
assert set(result.identify.admissible) == {"m_good", "m_weak", "m_groupy"}
assert set(result.identify.rejected) == {"m_wrong_sign", "m_noise"}

# Range is exactly the image of beta on survivors (min/max), nothing else.
L = min(result.identify.beta_values[m] for m in result.identify.admissible)
U = max(result.identify.beta_values[m] for m in result.identify.admissible)
assert abs(result.identify.range_L - L) < 1e-12
assert abs(result.identify.range_U - U) < 1e-12
assert result.identify.range_L < result.identify.range_U  # wide range: fragility is visible

# Anchors are excluded from the freeze preimage.
assert result_no_anchors.run_id == result.run_id
assert result_no_anchors.anchors_hash is None and result.anchors_hash is not None

# delta-grid is monotone: M*(d1) subset M*(d2) subset M*(d3) for d1<d2<d3.
d_sets = [set(r.admissible) for r in result.delta_grid.rows]
assert d_sets[0] <= d_sets[1] <= d_sets[2]
assert result.delta_grid.rows[0].delta_value == 0.0
assert len(d_sets[0]) == 3   # declared delta: designed valids only
assert len(d_sets[2]) == 5   # at delta=0.9 every menu measure passes

# theta-grid: tightening thresholds empties the set at lambda=2.5.
assert result.theta_grid.rows[-1].lambda_value == 2.5
assert result.theta_grid.rows[-1].empty is True
assert result.theta_grid.rows[-1].range_L is None

# Bootstrap band is present and the headline is untouched.
assert b.band_L is not None and b.band_U is not None
assert b.replicates_nonempty + b.replicates_empty + b.replicates_degenerate == b.n_boot

# CLI stdout parsed as JSON and agrees with the engine on the headline set.
assert cli["M_star"] == result.identify.admissible
assert abs(cli["L"] - result.identify.range_L) < 1e-12
assert cli["theta_grid"]["rows"] == 3 and cli["delta_grid"]["rows"] == 2

print("ALL ASSERTIONS PASSED")


## 10. The audit trail

Every run directory mirrors exactly the layers it produced:
`run_manifest.json` (freeze + hashes), `score_manifest.json`, `S_frozen.csv/.parquet`,
`network_resolved.json`, `beta_resolved.json`, `slacks.csv`, `admissible.json`,
`beta_values.json`, `range.json`, `report.html` + `report.json`, plus `bootstrap.json`,
`theta_grid.json`, `delta_grid.json`, and `anchors.json` when those layers are on.


In [ ]:
print("run dir contents (full-diagnostics run):")
for p in sorted(result.out_dir.iterdir()):
    print("  ", p.name)


## Recap

- **Admissible set** `M*` = measures that survive every researcher-authored restriction.
- **Construct-identified range** `[L,U]` = image of the target functional on survivors only.
- **Diagnostics are additive**: bootstrap band, θ-grid, δ-grid, and anchors never move the headline.
- **Empty sets and wide ranges are findings**, and the engine says so in the report.
- The engine is score-agnostic and model-free: the menu, the network, and β are researcher-owned.
